# Conjunto de datos completo sin clusterización

In [1]:
#Importaciones
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

#Lectura de datos
datos = pd.read_excel('03_Clusterizacion.xlsx')
datos.head(24)

,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM
0,2022-09-01 00:00:00,0.000000,19,7,77,0,4,15,0,Noche,Noche
1,2022-09-01 01:00:00,0.000000,19,7,82,0,4,16,1,Noche,Noche
2,2022-09-01 02:00:00,0.000000,18,9,85,0,3,16,2,Noche,Noche
3,2022-09-01 03:00:00,0.000000,18,11,87,0,3,16,3,Noche,Noche
4,2022-09-01 04:00:00,0.000000,18,11,88,0,3,16,4,Noche,Noche
5,2022-09-01 05:00:00,0.000000,17,15,86,0,3,14,5,Noche,Noche
6,2022-09-01 06:00:00,0.000000,18,47,89,0,3,16,6,Nublado,Lluvioso
7,2022-09-01 07:00:00,6.584959,18,51,95,0,4,17,7,Nublado,Lluvioso
8,2022-09-01 08:00:00,560.422022,18,47,100,0,3,18,8,Nublado,Lluvioso
9,2022-09-01 09:00:00,7720.582326,18,5,100,1,4,18,9,Nublado,Lluvioso


In [ ]:
datos["Generacion_prev_hour"] = datos["Generación"].shift(1)
datos["Generacion_prev_day"] = datos["Generación"].shift(24)
datos["Generacion_prev_year"] = datos["Generación"].shift(365)
datos = datos.dropna(how="any", axis= 0)

Definimos X y y

In [3]:
datos_dia = datos.copy()
datos_dia.head(10)

,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day,Generacion_prev_year
365,2022-09-16 05:00:00,0.000000,17,7,89,0,4,15,5,Noche,Noche,0.000000,0.000000,0.000000
366,2022-09-16 06:00:00,0.000000,15,7,90,0,4,14,6,Nublado,Lluvioso,0.000000,0.000000,0.000000
367,2022-09-16 07:00:00,0.000000,16,7,92,0,4,15,7,Nublado,Lluvioso,0.000000,0.000000,0.000000
368,2022-09-16 08:00:00,549.171616,16,7,95,0,3,15,8,Nublado,Lluvioso,0.000000,598.782293,0.000000
369,2022-09-16 09:00:00,14143.959531,17,7,92,1,4,15,9,Nublado,Lluvioso,549.171616,3712.946547,0.000000
370,2022-09-16 10:00:00,22469.850267,17,7,85,2,4,15,10,Nublado,Lluvioso,14143.959531,18095.003517,0.000000
371,2022-09-16 11:00:00,20066.612609,18,6,76,3,4,14,11,Nublado,Lluvioso,22469.850267,23644.907850,0.000000
372,2022-09-16 12:00:00,4688.612553,19,5,70,5,4,14,12,Nublado,Lluvioso,20066.612609,19925.949466,6.584959
373,2022-09-16 13:00:00,20911.327709,20,5,68,7,4,14,13,Nublado,Lluvioso,4688.612553,27000.000000,560.422022
374,2022-09-16 14:00:00,14667.376184,21,8,65,7,4,14,14,Nublado,Lluvioso,20911.327709,21306.124361,7720.582326


In [4]:
columns = datos_dia.drop(columns=["Fecha", "Generación", "Cluster KMeans", "Cluster GMM"]).columns

In [5]:
X = datos_dia[columns]
X

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day,Generacion_prev_year
365,17,7,89,0,4,15,5,0.000000,0.000000,0.0
366,15,7,90,0,4,14,6,0.000000,0.000000,0.0
367,16,7,92,0,4,15,7,0.000000,0.000000,0.0
368,16,7,95,0,3,15,8,0.000000,598.782293,0.0
369,17,7,92,1,4,15,9,549.171616,3712.946547,0.0
...,...,...,...,...,...,...,...,...,...,...
18285,22,0,45,0,1,9,20,1450.000000,0.000000,27421.0
18286,20,0,54,0,1,10,21,0.000000,0.000000,26234.0
18287,18,0,62,0,1,11,22,0.000000,0.000000,24939.0
18288,17,0,69,0,1,11,23,0.000000,0.000000,15438.0


In [6]:
y = datos_dia[['Generación']]
y

,Generación
365,0.000000
366,0.000000
367,0.000000
368,549.171616
369,14143.959531
...,...
18285,0.000000
18286,0.000000
18287,0.000000
18288,0.000000


Dividimos entrenamiento, validación y prueba

In [7]:
train_size = int(0.7 * len(X))
val_size = int(0.85 * len(X))

In [8]:
# Entrenamiento, validación y prueba, 75, 15 y 15
X_train, y_train =  X.iloc[:train_size, :], y.iloc[:train_size, :]
X_val, y_val = X.iloc[train_size:val_size, :], y.iloc[train_size:val_size, :]
X_test, y_test = X.iloc[val_size:, :],  y.iloc[val_size:,:]

print(f'X_train: {len(X_train)}, y_train: {len(y_train)}')
print(f'X_val: {len(X_val)}, y_val: {len(y_val)}')
print(f'X_test: {len(X_test)}, y_test: {len(y_test)}')

X_train: 12547, y_train: 12547
X_val: 2689, y_val: 2689
X_test: 2689, y_test: 2689


## Escalar con MinMaxScaler

In [9]:
from sklearn.preprocessing import MinMaxScaler

In [10]:
x_scaler = MinMaxScaler().fit(X_train)
x_scaler

MinMaxScaler()

In [11]:
X_train_scaled = x_scaler.transform(X_train)
print(X_train_scaled)
print(X_train_scaled.shape)

[[0.44736842 0.07777778 0.88421053 ... 0.         0.         0.        ]
 [0.39473684 0.07777778 0.89473684 ... 0.         0.         0.        ]
 [0.42105263 0.07777778 0.91578947 ... 0.         0.         0.        ]
 ...
 [0.57894737 0.         0.21052632 ... 0.04676667 0.         1.        ]
 [0.5        0.         0.29473684 ... 0.         0.         1.        ]
 [0.47368421 0.         0.35789474 ... 0.         0.         1.        ]]
(12547, 10)


In [12]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
X_train_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day,Generacion_prev_year
365,0.447368,0.077778,0.884211,0.000000,0.75,0.789474,0.217391,0.000000,0.000000,0.0
366,0.394737,0.077778,0.894737,0.000000,0.75,0.736842,0.260870,0.000000,0.000000,0.0
367,0.421053,0.077778,0.915789,0.000000,0.75,0.789474,0.304348,0.000000,0.000000,0.0
368,0.421053,0.077778,0.947368,0.000000,0.50,0.789474,0.347826,0.000000,0.019959,0.0
369,0.447368,0.077778,0.915789,0.071429,0.75,0.789474,0.391304,0.018306,0.123765,0.0
...,...,...,...,...,...,...,...,...,...,...
12907,0.710526,0.000000,0.105263,0.071429,0.00,0.052632,0.782609,0.768333,0.456633,1.0
12908,0.657895,0.000000,0.147368,0.000000,0.00,0.000000,0.826087,0.466667,0.048100,1.0
12909,0.578947,0.000000,0.210526,0.000000,0.00,0.052632,0.869565,0.046767,0.000000,1.0
12910,0.500000,0.000000,0.294737,0.000000,0.00,0.157895,0.913043,0.000000,0.000000,1.0


In [13]:
X_val_scaled = x_scaler.transform(X_val)
print(X_val_scaled)
print(X_val_scaled.shape)

[[0.42105263 0.         0.41052632 ... 0.         0.         1.        ]
 [0.36842105 0.         0.46315789 ... 0.         0.         0.2294    ]
 [0.34210526 0.         0.51578947 ... 0.         0.         0.        ]
 ...
 [0.71052632 0.01111111 0.34736842 ... 0.044      0.         0.80986667]
 [0.68421053 0.01111111 0.41052632 ... 0.         0.         0.67996667]
 [0.63157895 0.01111111 0.49473684 ... 0.         0.         0.50303333]]
(2689, 10)


In [14]:
X_val_scaled_df = pd.DataFrame(X_val_scaled, index=X_val.index, columns=X_val.columns)
X_val_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day,Generacion_prev_year
12912,0.421053,0.000000,0.410526,0.000000,0.00,0.210526,1.000000,0.000000,0.000000,1.000000
12913,0.368421,0.000000,0.463158,0.000000,0.00,0.210526,0.000000,0.000000,0.000000,0.229400
12914,0.342105,0.000000,0.515789,0.000000,0.00,0.210526,0.043478,0.000000,0.000000,0.000000
12915,0.315789,0.000000,0.526316,0.000000,0.00,0.157895,0.086957,0.000000,0.000000,0.000000
12916,0.315789,0.000000,0.526316,0.000000,0.00,0.157895,0.130435,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...
15596,0.789474,0.000000,0.252632,0.071429,0.25,0.578947,0.826087,0.712800,0.355167,0.844833
15597,0.763158,0.011111,0.284211,0.000000,0.25,0.578947,0.869565,0.326967,0.046733,0.964167
15598,0.710526,0.011111,0.347368,0.000000,0.00,0.631579,0.913043,0.044000,0.000000,0.809867
15599,0.684211,0.011111,0.410526,0.000000,0.00,0.684211,0.956522,0.000000,0.000000,0.679967


In [15]:
X_test_scaled = x_scaler.transform(X_test)
print(X_test_scaled)
print(X_test_scaled.shape)

[[0.57894737 0.01111111 0.58947368 ... 0.         0.         0.11496667]
 [0.57894737 0.01111111 0.6        ... 0.         0.         0.00816667]
 [0.55263158 0.02222222 0.65263158 ... 0.         0.         0.        ]
 ...
 [0.47368421 0.         0.6        ... 0.         0.         0.8313    ]
 [0.44736842 0.         0.67368421 ... 0.         0.         0.5146    ]
 [0.42105263 0.         0.71578947 ... 0.         0.         0.0567    ]]
(2689, 10)


In [16]:
X_test_scaled_df = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)
X_test_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day,Generacion_prev_year
15601,0.578947,0.011111,0.589474,0.0,0.0,0.789474,0.000000,0.000000,0.0,0.114967
15602,0.578947,0.011111,0.600000,0.0,0.0,0.736842,0.043478,0.000000,0.0,0.008167
15603,0.552632,0.022222,0.652632,0.0,0.0,0.736842,0.086957,0.000000,0.0,0.000000
15604,0.526316,0.022222,0.715789,0.0,0.0,0.789474,0.130435,0.000000,0.0,0.000000
15605,0.500000,0.022222,0.768421,0.0,0.0,0.789474,0.173913,0.000000,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...
18285,0.578947,0.000000,0.421053,0.0,0.0,0.473684,0.869565,0.048333,0.0,0.914033
18286,0.526316,0.000000,0.515789,0.0,0.0,0.526316,0.913043,0.000000,0.0,0.874467
18287,0.473684,0.000000,0.600000,0.0,0.0,0.578947,0.956522,0.000000,0.0,0.831300
18288,0.447368,0.000000,0.673684,0.0,0.0,0.578947,1.000000,0.000000,0.0,0.514600


In [17]:
x_scaller_all = MinMaxScaler().fit(X)
print(x_scaller_all)

MinMaxScaler()


In [18]:
X_scaled = x_scaller_all.transform(X)
print(X_scaled)
print(X_scaled.shape)

[[0.43589744 0.07777778 0.88659794 ... 0.         0.         0.        ]
 [0.38461538 0.07777778 0.89690722 ... 0.         0.         0.        ]
 [0.41025641 0.07777778 0.91752577 ... 0.         0.         0.        ]
 ...
 [0.46153846 0.         0.60824742 ... 0.         0.         0.8313    ]
 [0.43589744 0.         0.68041237 ... 0.         0.         0.5146    ]
 [0.41025641 0.         0.72164948 ... 0.         0.         0.0567    ]]
(17925, 10)


In [19]:
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)
X_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day,Generacion_prev_year
365,0.435897,0.077778,0.886598,0.000000,0.75,0.75,0.217391,0.000000,0.000000,0.000000
366,0.384615,0.077778,0.896907,0.000000,0.75,0.70,0.260870,0.000000,0.000000,0.000000
367,0.410256,0.077778,0.917526,0.000000,0.75,0.75,0.304348,0.000000,0.000000,0.000000
368,0.410256,0.077778,0.948454,0.000000,0.50,0.75,0.347826,0.000000,0.019959,0.000000
369,0.435897,0.077778,0.917526,0.071429,0.75,0.75,0.391304,0.018306,0.123765,0.000000
...,...,...,...,...,...,...,...,...,...,...
18285,0.564103,0.000000,0.432990,0.000000,0.00,0.45,0.869565,0.048333,0.000000,0.914033
18286,0.512821,0.000000,0.525773,0.000000,0.00,0.50,0.913043,0.000000,0.000000,0.874467
18287,0.461538,0.000000,0.608247,0.000000,0.00,0.55,0.956522,0.000000,0.000000,0.831300
18288,0.435897,0.000000,0.680412,0.000000,0.00,0.55,1.000000,0.000000,0.000000,0.514600


In [20]:
y_scaler = MinMaxScaler().fit(y_train)
print(y_scaler)

MinMaxScaler()


In [21]:
y_train_scaled = y_scaler.transform(y_train)
print(y_train_scaled)
print(y_train_scaled.shape)

[[0.]
 [0.]
 [0.]
 ...
 [0.]
 [0.]
 [0.]]
(12547, 1)


In [22]:
y_train_scaled_df = pd.DataFrame(y_train_scaled, index=y_train.index, columns=y_train.columns)
y_train_scaled_df

,Generación
365,0.000000
366,0.000000
367,0.000000
368,0.018306
369,0.471465
...,...
12907,0.466667
12908,0.046767
12909,0.000000
12910,0.000000


In [23]:
y_val_scaled = y_scaler.transform(y_val)
print(y_val_scaled)
print(y_val_scaled.shape)

[[0.]
 [0.]
 [0.]
 ...
 [0.]
 [0.]
 [0.]]
(2689, 1)


In [24]:
y_val_scaled_df = pd.DataFrame(y_val_scaled, index=y_val.index, columns=y_val.columns)
y_val_scaled_df

,Generación
12912,0.000000
12913,0.000000
12914,0.000000
12915,0.000000
12916,0.000000
...,...
15596,0.326967
15597,0.044000
15598,0.000000
15599,0.000000


In [25]:
y_test_scaled = y_scaler.transform(y_test)
print(y_test_scaled)
print(y_test_scaled.shape)

[[0.]
 [0.]
 [0.]
 ...
 [0.]
 [0.]
 [0.]]
(2689, 1)


In [26]:
y_test_scaled_df = pd.DataFrame(y_test_scaled, index=y_test.index, columns=y_test.columns)
y_test_scaled_df

,Generación
15601,0.0
15602,0.0
15603,0.0
15604,0.0
15605,0.0
...,...
18285,0.0
18286,0.0
18287,0.0
18288,0.0


In [27]:
y_scaller_all = MinMaxScaler().fit(y)
print(y_scaller_all)

MinMaxScaler()


In [28]:
y_scaled = y_scaller_all.transform(y)
print(y_scaled)
print(y_scaled.shape)

[[0.]
 [0.]
 [0.]
 ...
 [0.]
 [0.]
 [0.]]
(17925, 1)


In [29]:
y_scaled_df = pd.DataFrame(y_scaled, index=y.index, columns=y.columns)
y_scaled_df

,Generación
365,0.000000
366,0.000000
367,0.000000
368,0.018306
369,0.471465
...,...
18285,0.000000
18286,0.000000
18287,0.000000
18288,0.000000


## Definición de modelos

### LightGBM

In [30]:
from lightgbm import LGBMRegressor
import optuna
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import seaborn as sns
from sklearn.metrics import mean_absolute_percentage_error as mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error as mean_absolute_error
from sklearn.metrics import mean_squared_error as mean_squared_error
from sklearn.metrics import r2_score as r2_score

In [31]:
# Inicializar listas para métricas
LightGBM_model = LGBMRegressor(num_leaves=500, subsample= 0.10698460631792395, colsample_bytree= 0.7272836809565294, min_data_in_leaf= 85)
LightGBM_model.fit(X_train_scaled_df, y_train_scaled_df)
resultados = pd.DataFrame(index = y_test_scaled_df.index, columns=["LightGBM"])
#Ciclo diario de predicción
for i in range(len(X_test)):
    inicio = i * 1
    fin = inicio + 1

    X_test_seg = X_test_scaled_df.iloc[inicio:fin, :]
    y_test_seg = y_test_scaled_df.iloc[inicio:fin]

    if len(X_test_seg) < 1:
        break

    y_pred = LightGBM_model.predict(X_test_seg)
    y_pred = y_scaler.inverse_transform(y_pred.reshape(-1, 1))
    y_pred = np.clip(y_pred, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

    resultados.iloc[i, 0] = y_pred[0, 0]

[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000171 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1029
[LightGBM] [Info] Number of data points in the train set: 12547, number of used features: 10
[LightGBM] [Info] Start training from score 0.296076
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with p

In [32]:
resultados

,LightGBM
15601,0.0
15602,0.0
15603,0.0
15604,0.0
15605,0.0
...,...
18285,0.0
18286,9.150655
18287,6.915859
18288,0.0


In [33]:
predicciones = y_test.copy()
predicciones

,Generación
15601,0.0
15602,0.0
15603,0.0
15604,0.0
15605,0.0
...,...
18285,0.0
18286,0.0
18287,0.0
18288,0.0


In [34]:
predicciones["LightGBM"] = resultados["LightGBM"]
predicciones

,Generación,LightGBM
15601,0.0,0.0
15602,0.0,0.0
15603,0.0,0.0
15604,0.0,0.0
15605,0.0,0.0
...,...,...
18285,0.0,0.0
18286,0.0,9.150655
18287,0.0,6.915859
18288,0.0,0.0


In [35]:
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['LightGBM'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['LightGBM']):.4f}")

MAE: 1018.6100
RMSE: 2112.2661
R²: 0.9666


## Random Forest

In [36]:
from sklearn.ensemble import RandomForestRegressor

In [37]:
#Modelo LightGBM
RF_model = RandomForestRegressor(
    criterion="squared_error",
    random_state=0,
    n_estimators=400,
    min_impurity_decrease=0,
    max_depth=None,
    bootstrap=True
)
RF_model.fit(X_train_scaled_df, y_train_scaled_df)
# Inicializar listas para métricas
resultados = pd.DataFrame(index = y_test_scaled_df.index, columns=["Random Forest"])
#Ciclo diario de predicción
for i in range(len(X_test)):
    inicio = i * 1
    fin = inicio + 1

    X_test_seg = X_test_scaled_df.iloc[inicio:fin, :]
    y_test_seg = y_test_scaled_df.iloc[inicio:fin]

    if len(X_test_seg) < 1:
        break

    y_pred = RF_model.predict(X_test_seg)
    y_pred = y_scaler.inverse_transform(y_pred.reshape(-1, 1))
    y_pred = np.clip(y_pred, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

    resultados.iloc[i, 0] = y_pred[0, 0]

In [ ]:
import optuna

# Función objetivo para Optuna
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 5, 50),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
        'random_state': 42
    }

    model = RandomForestRegressor(**params)
    scores = cross_val_score(model, X_train_scaled_df, y_train_scaled_df, cv=5, scoring='neg_root_mean_squared_error')
    return -scores.mean()

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50, n_jobs=-1)

print(f"Mejores hiperparámetros encontrados: {study.best_params}")


[I 2025-03-09 17:50:36,820] A new study created in memory with name: no-name-4e022f9e-44e2-4437-a7bc-0ba7b23c7154
[I 2025-03-09 17:51:16,668] Trial 3 finished with value: 0.07858532801037847 and parameters: {'n_estimators': 247, 'max_depth': 44, 'min_samples_split': 17, 'min_samples_leaf': 8, 'max_features': 'sqrt', 'bootstrap': False}. Best is trial 3 with value: 0.07858532801037847.
[I 2025-03-09 17:51:41,411] Trial 0 finished with value: 0.08032450774045893 and parameters: {'n_estimators': 546, 'max_depth': 46, 'min_samples_split': 7, 'min_samples_leaf': 8, 'max_features': 'log2', 'bootstrap': True}. Best is trial 3 with value: 0.07858532801037847.
[I 2025-03-09 17:52:11,337] Trial 2 finished with value: 0.07863526994634071 and parameters: {'n_estimators': 717, 'max_depth': 42, 'min_samples_split': 4, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'bootstrap': True}. Best is trial 3 with value: 0.07858532801037847.


In [38]:
predicciones["Random Forest"] = resultados["Random Forest"]
predicciones

,Generación,LightGBM,Random Forest
15601,0.0,0.0,0.0
15602,0.0,0.0,0.0
15603,0.0,0.0,0.0
15604,0.0,0.0,0.0
15605,0.0,0.0,0.0
...,...,...,...
18285,0.0,0.0,0.0
18286,0.0,9.150655,0.0
18287,0.0,6.915859,0.0
18288,0.0,0.0,0.0


## CTNET

## Métricas

In [39]:
print("LightGBM")
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['LightGBM'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print("Random Forest")
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['Random Forest']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['Random Forest'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['Random Forest']):.4f}")

LightGBM
MAE: 1018.6100
RMSE: 2112.2661
R²: 0.9666
Random Forest
MAE: 994.1063
RMSE: 2219.3288
R²: 0.9632
